# Ropedia Academy — C9 · Paper deep-dive & reproducing a model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/C9.ipynb)

> **Builds the simplest honest baseline (frozen features + linear head), reports top-1/5 on a held-out split, and runs a shuffle-control ablation to prove the metric is real.**
>
> 搭出最简诚实基线（冻结特征 + 线性头部），在留出划分上报告 top-1/5，并跑打乱对照消融以证明指标可信。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/C9

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Honest baseline: FROZEN backbone features + a small trained head, on a held-out split.
torch.manual_seed(0)
N, D, K = 600, 256, 20
labels = torch.randint(0, K, (N,))
feats  = torch.randn(N, D) + F.one_hot(labels, K).float() @ torch.randn(K, D)   # signal+noise
tr, te = slice(0, 500), slice(500, N)               # split by index (a real one: by participant)

def train_head(x):
    head = nn.Linear(D, K); opt = torch.optim.Adam(head.parameters(), 1e-2)
    for _ in range(300):
        opt.zero_grad(); F.cross_entropy(head(x[tr]), labels[tr]).backward(); opt.step()
    return head

head = train_head(feats)
topk = lambda k: (head(feats[te]).topk(k,-1).indices == labels[te][:,None]).any(-1).float().mean().item()
print(f"held-out  top-1={topk(1):.2f}  top-5={topk(5):.2f}")

# ABLATION control: destroy the signal -> accuracy must fall to chance (1/K).
shuf = train_head(feats[torch.randperm(N)])
acc = (shuf(feats[te]).argmax(-1) == labels[te]).float().mean().item()
print(f"shuffled-feature control top-1={acc:.2f} (~chance {1/K:.2f}) -> the metric is trustworthy")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/C9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks